# Chapter 13 — Binomial Trees

Code behind every number in the chapter. **Standard library only** — the whole
chapter is arithmetic on a lattice, and writing it without NumPy keeps each step
visible rather than vectorized away.

| Section | What it reproduces |
|---|---|
| 1 | The one-step tree, the hedge, and the arbitrage (Example 13.1) |
| 2 | The real-world route, and why it is circular (Example 13.2) |
| 3 | State prices (Example 13.3) |
| 4 | Three states, and the price interval (Example 13.4) |
| 5 | Two steps, and following the hedge through (Example 13.5) |
| 6 | An American put and the early-exercise premium (Example 13.6) |
| 7 | Building a tree from a volatility (Example 13.7) |
| 8 | An American call on futures, exercised early (Example 13.8) |
| 9 | Convergence, and the Black–Scholes–Merton limit (Figure 13.6) |
| 10 | The early-exercise boundary (Figure 13.5) |

Each cell prints the computed value beside the one the chapter prints, so a
figure that has drifted is visible rather than merely absent.

In [1]:
from math import exp, sqrt


def show(label, got, printed, tol=6e-4):
    """Print a computed value beside the one the chapter prints."""
    flag = "ok" if abs(got - printed) <= tol * max(1.0, abs(printed)) else "DIFFERS"
    print("%-44s %12.6f   book %10.4f   %s" % (label, got, printed, flag))


S0, Su, Sd, K, r, dt = 20.0, 22.0, 18.0, 21.0, 0.04, 0.25
print("one-step tree: S0=%.0f -> {%.0f, %.0f}, K=%.0f, r=%.0f%%, dt=%.2f"
      % (S0, Su, Sd, K, 100 * r, dt))

one-step tree: S0=20 -> {22, 18}, K=21, r=4%, dt=0.25


## 1. One step, and the hedge that prices it — Example 13.1

The delta is chosen so the hedged portfolio pays the same in both states. Once it
does, the portfolio is riskless and must earn $r$ — and no probability has been
used. Writing the call at \$0.60 instead of its value leaves a profit in *every*
state, which is what makes it an arbitrage rather than a good bet.

In [2]:
fu, fd = max(Su - K, 0.0), max(Sd - K, 0.0)
delta = (fu - fd) / (Su - Sd)
cash = (fd - delta * Sd) * exp(-r * dt)          # borrowing that completes the hedge
f = delta * S0 + cash

show("delta", delta, 0.25)
show("call value f", f, 0.545)

# the replicating portfolio pays what the call pays, in both states
up = delta * Su + cash * exp(r * dt)
dn = delta * Sd + cash * exp(r * dt)
show("replicating portfolio, up state", up, fu)
show("replicating portfolio, down state", dn, fd)

quote = 0.60
show("arbitrage profit at a %.2f quote" % quote, (quote - f) * exp(r * dt), 0.056)

delta                                            0.250000   book     0.2500   ok
call value f                                     0.544776   book     0.5450   ok
replicating portfolio, up state                  1.000000   book     1.0000   ok
replicating portfolio, down state                0.000000   book     0.0000   ok
arbitrage profit at a 0.60 quote                 0.055779   book     0.0560   ok


## 2. The real-world route — Example 13.2

Done properly once, to show why it is abandoned. The true probability gives an
expected payoff easily enough; the discount rate it needs is the one thing the
calculation cannot supply, because knowing it requires the option price we set
out to find.

In [3]:
q_real = 0.7
ES = q_real * Su + (1 - q_real) * Sd
show("expected stock price", ES, 20.80)
show("expected return over the step, %", 100 * (ES / S0 - 1), 4.0)
show("expected call payoff", q_real * fu + (1 - q_real) * fd, 0.70)

# the required return the option would need, once the answer is already known.
# the chapter divides by the rounded 0.545, so do the same and show both.
EF = q_real * fu + (1 - q_real) * fd
show("implied required return, on f = 0.545, %", 100 * (EF / 0.545 - 1), 28.4, tol=3e-3)
print("%-44s %12.6f" % ("the same on the unrounded f", 100 * (EF / f - 1)))

expected stock price                            20.800000   book    20.8000   ok
expected return over the step, %                 4.000000   book     4.0000   ok
expected call payoff                             0.700000   book     0.7000   ok
implied required return, on f = 0.545, %        28.440367   book    28.4000   ok
the same on the unrounded f                     28.493238


## 3. State prices — Example 13.3

$\psi_u$ and $\psi_d$ are the prices today of \$1 in each state. They are the
discounted risk-neutral probabilities, and once they exist every claim on this
tree is priced by a dot product with no further argument.

In [4]:
p = (exp(r * dt) - Sd / S0) / (Su / S0 - Sd / S0)
psi_u = exp(-r * dt) * p
psi_d = exp(-r * dt) * (1 - p)
show("risk-neutral probability p", p, 0.5503)
show("state price psi_u", psi_u, 0.5448)
show("state price psi_d", psi_d, 0.4453)

# verify psi_u by replicating $1 in the up state
d_rep = (1.0 - 0.0) / (Su - Sd)
b_rep = (0.0 - d_rep * Sd) * exp(-r * dt)
show("B in that replication", b_rep, -4.4552)
show("cost today, which is psi_u", d_rep * S0 + b_rep, 0.5448)

# and now price something else with no further work
show("claim paying 3 up / 7 down", 3 * psi_u + 7 * psi_d, 4.7512)

risk-neutral probability p                       0.550251   book     0.5503   ok
state price psi_u                                0.544776   book     0.5448   ok
state price psi_d                                0.445274   book     0.4453   ok
B in that replication                           -4.455224   book    -4.4552   ok
cost today, which is psi_u                       0.544776   book     0.5448   ok
claim paying 3 up / 7 down                       4.751246   book     4.7512   ok


## 4. Three states — Example 13.4

Add a middle state and the replication fails: three equations, two unknowns. What
survives is a *range* of prices consistent with no arbitrage, which is the second
fundamental theorem arriving early and in miniature.

In [5]:
# psi must price the bond and the stock, and stay positive
#   psi1 + psi2 + psi3 = exp(-r dt)          (the bond)
#   22 psi1 + 20 psi2 + 18 psi3 = 20         (the stock)
# one free parameter remains; sweep it and read off the call's price range
lo, hi = None, None
steps = 200000
for i in range(1, steps):
    psi2 = exp(-r * dt) * i / steps
    psi1 = (20.0 - 18.0 * (exp(-r * dt) - psi2) - 20.0 * psi2) / (22.0 - 18.0)
    psi3 = exp(-r * dt) - psi1 - psi2
    if psi1 > 0 and psi2 > 0 and psi3 > 0:
        c = 1.0 * psi1                        # the call pays $1 only in the top state
        lo = c if lo is None else min(lo, c)
        hi = c if hi is None else max(hi, c)
print("no-arbitrage price range for the call: %.4f to %.4f" % (lo, hi))
print("the replicating system is inconsistent, so no single price follows")

no-arbitrage price range for the call: 0.0995 to 0.5448
the replicating system is inconsistent, so no single price follows


## 5. Two steps, and following the hedge — Example 13.5

Backward induction gives the price; the point of the example is that the hedge
actually works. The trader writes the call, buys $\Delta_A$ shares, borrows the
rest, and rebalances once — and arrives with exactly the payoff owed.

In [6]:
u, d = 1.1, 0.9
p2 = (exp(r * dt) - d) / (u - d)
S = {"A": S0, "B": S0 * u, "C": S0 * d,
     "BB": S0 * u * u, "BC": S0 * u * d, "CC": S0 * d * d}
f_BB, f_BC, f_CC = (max(S[k] - K, 0.0) for k in ("BB", "BC", "CC"))
f_B = exp(-r * dt) * (p2 * f_BB + (1 - p2) * f_BC)
f_C = exp(-r * dt) * (p2 * f_BC + (1 - p2) * f_CC)
f_A = exp(-r * dt) * (p2 * f_B + (1 - p2) * f_C)

show("payoff at BB", f_BB, 3.20)
show("value at node B", f_B, 1.7433)
show("value at node A", f_A, 0.9497)

dA = (f_B - f_C) / (S["B"] - S["C"])
dB = (f_BB - f_BC) / (S["BB"] - S["BC"])
show("delta at A", dA, 0.4358)
show("delta at B", dB, 0.7273)

borrow = dA * S["A"] - f_A
show("borrowed at A", borrow, 7.7667)
show("debt one step later", borrow * exp(r * dt), 7.8448)
show("shares worth, at node B", dA * S["B"], 9.5881)

payoff at BB                                     3.200000   book     3.2000   ok
value at node B                                  1.743282   book     1.7433   ok
value at node A                                  0.949698   book     0.9497   ok
delta at A                                       0.435821   book     0.4358   ok
delta at B                                       0.727273   book     0.7273   ok
borrowed at A                                    7.766714   book     7.7667   ok
debt one step later                              7.844771   book     7.8448   ok
shares worth, at node B                          9.588053   book     9.5881   ok


## 6. An American put — Example 13.6

At each node the value is the larger of continuing and exercising. Exercise is
optimal at node C and nowhere else, which both prices the option and *locates*
the exercise boundary — the practical advantage a tree has over a formula.

In [7]:
S0p, Kp, rp, up_, dp_, dtp = 50.0, 52.0, 0.05, 1.2, 0.8, 1.0
disc = exp(-rp * dtp)
pp = (exp(rp * dtp) - dp_) / (up_ - dp_)
show("exp(r dt)", exp(rp * dtp), 1.051271)
show("p", pp, 0.6282)

pay = lambda s: max(Kp - s, 0.0)
D, E, F = pay(S0p * up_ * up_), pay(S0p * up_ * dp_), pay(S0p * dp_ * dp_)
print("terminal payoffs D, E, F = %.0f, %.0f, %.0f" % (D, E, F))

cont_B = disc * (pp * D + (1 - pp) * E)
val_B = max(cont_B, pay(S0p * up_))
show("node B continuation", cont_B, 1.4148)

cont_C = disc * (pp * E + (1 - pp) * F)
val_C = max(cont_C, pay(S0p * dp_))
show("node C continuation", cont_C, 9.4639)
show("node C value (exercise wins)", val_C, 12.0)

amer = disc * (pp * val_B + (1 - pp) * val_C)
euro = disc * (pp * cont_B + (1 - pp) * cont_C)
show("American put", amer, 5.0896)
show("European put", euro, 4.1927)
show("early exercise premium", amer - euro, 0.897)

exp(r dt)                                        1.051271   book     1.0513   ok
p                                                0.628178   book     0.6282   ok
terminal payoffs D, E, F = 0, 4, 20
node B continuation                              1.414753   book     1.4148   ok
node C continuation                              9.463930   book     9.4639   ok
node C value (exercise wins)                    12.000000   book    12.0000   ok
American put                                     5.089632   book     5.0896   ok
European put                                     4.192654   book     4.1927   ok
early exercise premium                           0.896978   book     0.8970   ok


## 7. A tree from a volatility — Example 13.7

The Cox–Ross–Rubinstein choice $u = e^{\sigma\sqrt{\Delta t}}$, $d = 1/u$. Two
checks are worth running every time: $0 < p < 1$, and $d < e^{r\Delta t} < u$.
Failing either means the step is too long for the volatility, and the tree returns
negative probabilities rather than an error.

In [8]:
sig, dtc, rc = 0.30, 1.0 / 12, 0.04
uc = exp(sig * sqrt(dtc))
dc = 1.0 / uc
growth = exp(rc * dtc)
pc = (growth - dc) / (uc - dc)
show("u", uc, 1.09046)
show("d", dc, 0.91704)
show("exp(r dt)", growth, 1.003339)
show("p", pc, 0.49762)
print("0 < p < 1                : %s" % (0 < pc < 1))
print("d < exp(r dt) < u        : %s" % (dc < growth < uc))

# the chapter's first-order approximation, p ~ 1/2 + (1/2)((r - sigma^2/2)/sigma) sqrt(dt)
approx = 0.5 + 0.5 * ((rc - sig * sig / 2) / sig) * sqrt(dtc)
show("first-order approximation to p", approx, 0.4976)


u                                                1.090463   book     1.0905   ok
d                                                0.917042   book     0.9170   ok
exp(r dt)                                        1.003339   book     1.0033   ok
p                                                0.497616   book     0.4976   ok
0 < p < 1                : True
d < exp(r dt) < u        : True
first-order approximation to p                   0.497594   book     0.4976   ok


## 8. An American call on futures — Example 13.8

A futures contract costs nothing to enter, so $q = r$ and the early-exercise
argument of Section 11.8 loses its premise. Here the call *is* exercised early
with no dividend anywhere in the problem — a reader who remembered the slogan and
not its proof would price these wrongly.

In [9]:
F0, sigf, rf, dtf, Kf = 100.0, 0.25, 0.04, 0.25, 100.0
uf = exp(sigf * sqrt(dtf))
df_ = 1.0 / uf
pf = (1.0 - df_) / (uf - df_)                  # q = r for a futures underlying
discf = exp(-rf * dtf)
show("u", uf, 1.133148)
show("d", df_, 0.882497)
show("p", pf, 0.4688)

show("terminal F, up-up", F0 * uf * uf, 128.40)
show("terminal F, down-down", F0 * df_ * df_, 77.88)
show("payoff, up-up", max(F0 * uf * uf - Kf, 0.0), 28.4025)

cont_up = discf * (pf * max(F0 * uf * uf - Kf, 0.0))
exer_up = F0 * uf - Kf
show("upper node continuation", cont_up, 13.1824)
show("upper node exercise", exer_up, 13.3148)
print("exercise early at the upper node: %s" % (exer_up > cont_up))

val_up = max(cont_up, exer_up)
show("American call today", discf * (pf * val_up), 6.1798)
show("European call today", discf * (pf * cont_up), 6.1183)

u                                                1.133148   book     1.1331   ok
d                                                0.882497   book     0.8825   ok
p                                                0.468791   book     0.4688   ok
terminal F, up-up                              128.402542   book   128.4000   ok
terminal F, down-down                           77.880078   book    77.8800   ok
payoff, up-up                                   28.402542   book    28.4025   ok
upper node continuation                         13.182360   book    13.1824   ok
upper node exercise                             13.314845   book    13.3148   ok
exercise early at the upper node: True
American call today                              6.179767   book     6.1798   ok
European call today                              6.118277   book     6.1183   ok


## 9. Convergence, and the limit it converges to — Figure 13.6

The two-year put of the convergence figure, on a Cox–Ross–Rubinstein tree with
$n$ steps. The value settles toward the Black–Scholes–Merton limit that Chapter 15
derives, oscillating with the parity of $n$; the figure draws the limit as a dashed
line, and Chapter 15 quotes the American value the same tree gives when it is fine
enough.


In [10]:
from math import erf, log


def crr_put(S, K, r, sigma, T, n, american):
    """European or American put on an n-step Cox-Ross-Rubinstein tree."""
    dt = T / n
    u = exp(sigma * sqrt(dt)); d = 1.0 / u
    p = (exp(r * dt) - d) / (u - d); disc = exp(-r * dt)
    v = [max(K - S * u ** j * d ** (n - j), 0.0) for j in range(n + 1)]
    for i in range(n - 1, -1, -1):
        v = [disc * (p * v[j + 1] + (1 - p) * v[j]) for j in range(i + 1)]
        if american:
            v = [max(v[j], K - S * u ** j * d ** (i - j)) for j in range(i + 1)]
    return v[0]


def N(x):
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))


Sc, Kc, rr, sc, Tc = 50.0, 52.0, 0.05, 0.30, 2.0
show("European put, n = 1", crr_put(Sc, Kc, rr, sc, Tc, 1, False), 8.4503)
show("European put, n = 2", crr_put(Sc, Kc, rr, sc, Tc, 2, False), 6.2457)
show("European put, n = 50", crr_put(Sc, Kc, rr, sc, Tc, 50, False), 6.7872)
d1 = (log(Sc / Kc) + (rr + sc * sc / 2) * Tc) / (sc * sqrt(Tc))
d2 = d1 - sc * sqrt(Tc)
bs = Kc * exp(-rr * Tc) * N(-d2) - Sc * N(-d1)
show("Black-Scholes-Merton limit", bs, 6.7601)
show("American put on a 2000-step tree", crr_put(Sc, Kc, rr, sc, Tc, 2000, True), 7.4719)
print("odd and even n straddle each other, both above the limit at n = 50:",
      "%.4f (n = 49), %.4f (n = 50)" % (crr_put(Sc, Kc, rr, sc, Tc, 49, False), crr_put(Sc, Kc, rr, sc, Tc, 50, False)))


European put, n = 1                              8.450255   book     8.4503   ok
European put, n = 2                              6.245708   book     6.2457   ok
European put, n = 50                             6.787222   book     6.7872   ok
Black-Scholes-Merton limit                       6.760140   book     6.7601   ok


American put on a 2000-step tree                 7.471909   book     7.4719   ok
odd and even n straddle each other, both above the limit at n = 50: 6.7627 (n = 49), 6.7872 (n = 50)


## 10. The early-exercise boundary — Figure 13.5

The Example 13.6 put on a 1,000-step tree at the volatility of the two-step tree,
$\sigma = \ln(1.2/0.8)/2 = 20.3\%$. At each date the boundary is the price at which
exercising and continuing are worth the same, interpolated between the highest
node where exercise wins and the node above it. It starts just above 40, rises to about
42 by the one-year date, and climbs to the strike only in the final weeks: the
critical price $S^*$ of Figure 11.5, traced through time.


In [11]:
def exercise_boundary(S, K, r, sigma, T, n):
    """Dates -> price at which intrinsic value equals continuation value."""
    dt = T / n
    u = exp(sigma * sqrt(dt)); d = 1.0 / u
    p = (exp(r * dt) - d) / (u - d); disc = exp(-r * dt)
    grid = [[S * u ** j * d ** (i - j) for j in range(i + 1)] for i in range(n + 1)]
    v = [max(K - s, 0.0) for s in grid[n]]
    out = {}
    for i in range(n - 1, -1, -1):
        cont = [disc * (p * v[j + 1] + (1 - p) * v[j]) for j in range(i + 1)]
        gain = [K - grid[i][j] - cont[j] for j in range(i + 1)]      # exercise minus hold
        v = [max(c, K - s) for c, s in zip(cont, grid[i])]
        ex = [j for j in range(i + 1) if gain[j] >= 0 and K - grid[i][j] > 0]
        if ex:
            j = max(ex)
            if j + 1 <= i:
                s0, s1, g0, g1 = grid[i][j], grid[i][j + 1], gain[j], gain[j + 1]
                out[round(i * dt, 3)] = s0 + (s1 - s0) * g0 / (g0 - g1)
            else:
                out[round(i * dt, 3)] = grid[i][j]
    return out


bnd = exercise_boundary(50.0, 52.0, 0.05, log(1.2 / 0.8) / 2, 2.0, 1000)
show("boundary at t = 0.5", bnd[0.5], 41.0, tol=3e-3)
show("boundary at t = 1.0", bnd[1.0], 41.9, tol=3e-3)
show("boundary at t = 1.9", bnd[1.9], 46.8, tol=3e-3)
show("boundary at t = 1.98", bnd[1.98], 49.2, tol=3e-3)
print("the two-step tree exercised at 40 and held at 60 on the one-year date;",
      "the fine tree puts the boundary at %.2f" % bnd[1.0])


boundary at t = 0.5                             41.034991   book    41.0000   ok
boundary at t = 1.0                             41.939026   book    41.9000   ok
boundary at t = 1.9                             46.807278   book    46.8000   ok
boundary at t = 1.98                            49.212955   book    49.2000   ok
the two-step tree exercised at 40 and held at 60 on the one-year date; the fine tree puts the boundary at 41.94
